In [ ]:
import pandas as pd
from transformers import T5Tokenizer, Trainer, TrainingArguments, T5ForConditionalGeneration

In [ ]:
train_data = pd.read_csv("/content/samsum-train.csv")
val_data = pd.read_csv("/content/samsum-validation.csv")

In [ ]:
train_data = train_data.sample(n=4000, random_state=42).reset_index(drop=True)
val_data = val_data.sample(n=500, random_state=42).reset_index(drop=True)

In [ ]:
import re
def clean_data(text):
  text=re.sub(r"\r\n"," ",text)#lines replace
  text=re.sub(r"\s+", " ", text)#spaces
  text=re.sub(r"<.*?>", " ",text)#html tags
  text=text.strip().lower()
  return text

In [ ]:
train_data["dialogue"] = train_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

val_data["dialogue"] = val_data["dialogue"].apply(clean_data)
train_data["summary"] = train_data["summary"].apply(clean_data)

In [ ]:
tokenizer = T5Tokenizer.from_pretrained("t5-small")

In [ ]:
# raw data => tokenized inputs for fine tunning
def tokenize(data):
  inputs=tokenizer(data["dialogue"], padding="max_length", max_length=512,  truncation=True)
  targets=tokenizer(data["summary"], padding="max_length", max_length=150,  truncation=True)

  inputs["labels"] = targets["input_ids"]
  return inputs

In [ ]:
train_dataset =train_data.apply(tokenize, axis=1).tolist()
val_dataset =val_data.apply(tokenize, axis=1).tolist()

In [ ]:
model = T5ForConditionalGeneration.from_pretrained("t5-small")

In [ ]:
import torch
if torch.backends.mps.is_available():
  device=torch.device("mps")
elif torch.cuda.is_available():
  device=torch.device("cuda")
else:
  device=torch.device("cpu")

print("device:", device)
model.to(device)

In [ ]:
training_args=TrainingArguments(
    output_dir="./results",
    num_train_epochs=6,
    weight_decay=0.01,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    warmup_steps=500

)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

In [ ]:
trainer.train()